# 第 47 课｜Agent 评测：结果、轨迹、预算与失败归因

**学习项目：从 Python 到大模型原理、训练、推理、检索与 Agent。**

上一课完成代码修复循环。这一课把“展示一次成功”升级为可复核的评测：

**预先登记计划 → 独立运行每次试验 → 保存原始动作与工具结果 → 核对最终状态 → 固定分母汇总 → 比较改善与回归。**

本讲义与 `lesson_47_agent_evaluation_traces_and_budgets.py` 各自独立运行，不需要其他课的源码。

> 默认只有标准库、预写策略与内存服务。不会下载模型、创建真实工单、发送网络请求或执行候选代码。成绩是工程机制对照，不是 Qwen 或任何 LLM 的成绩。

## 0. 本课目标与使用方法

完成后，你应能区分 task / trial / 决策轮次 / 工具尝试，写出明确的成功条件；用计划任务数保留失败与缺失的分母；比较结果而非只看最终回复；解释 pass@k 与 pass^k 的差异；读取第 46 课记录并指出证据能证明到哪一步。

使用 Python 3.10+ 的现有 Jupyter 环境即可。按顺序运行，先预测输出，再展开需要研究的实现。带“完整实现”的长单元仅折叠显示，内容就在本 Notebook 中。

本课没有模型、GPU或外部API，因此 token、时延和费用未测时明确写 `None` / JSON `null`，不是零。独立练习检查默认关闭，不影响顺序运行。

In [1]:
import sys
from pathlib import Path
print("Python:", sys.version.split()[0])
print("工作目录:", Path.cwd())
print("依赖：Python标准库；不加载模型。")

Python: 3.13.5
工作目录: /mnt/data/_lesson47_build
依赖：Python标准库；不加载模型。


### 基础工具：固定格式、严格 JSON、内容摘要

下面定义文件读写与输入检查。重复键和 NaN 不会被静默接受；摘要只检查内容是否一致，不是签名或来源认证。JSON 输入还限制文件大小，不把任意文件自动反序列化成 Python 对象。

资料：[Python JSON 文档](https://docs.python.org/3/library/json.html)。

In [2]:
import argparse
import copy
import hashlib
import json
import math
import os
import platform
import random
import statistics
import sys
import tempfile
import uuid
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable

FORMAT = "lesson47-agent-evaluation-v1"
STRATEGIES = ("announce_only", "single_pass", "bounded_retry")
PROFILES = ("clean", "read_transient", "reply_lost", "calculate_transient")
BUDGET = {"max_turns": 8, "max_tool_attempts": 7, "retry_limit": 1}
SOURCES = {
    "agent_evals": "https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents",
    "tau_bench": "https://arxiv.org/abs/2406.12045",
    "pass_at_k": "https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py",
    "swe_bench": "https://www.swebench.com/SWE-bench/guides/evaluation/",
    "statistics": "https://docs.python.org/3/library/statistics.html",
    "json": "https://docs.python.org/3/library/json.html",
}


class ContractError(ValueError):
    """数据、轨迹或评测约定不满足。"""


def require(condition: bool, message: str) -> None:
    if not condition:
        raise ContractError(message)


def canonical(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True,
                      separators=(",", ":"), allow_nan=False)


def digest(value: Any) -> str:
    return hashlib.sha256(canonical(value).encode("utf-8")).hexdigest()


def source_digest(source: str) -> str:
    return hashlib.sha256(source.encode("utf-8")).hexdigest()


def strict_json(text: str) -> Any:
    require(type(text) is str and len(text.encode("utf-8")) <= 8_000_000, "JSON类型或大小无效")
    def pairs(items):
        result = {}
        for key, value in items:
            require(key not in result, "重复JSON键：" + key)
            result[key] = value
        return result
    def bad(value):
        raise ContractError("不允许NaN/Infinity：" + value)
    def number(value):
        result = float(value)
        require(math.isfinite(result), "非有限浮点数")
        return result
    try:
        return json.loads(text, object_pairs_hook=pairs, parse_constant=bad, parse_float=number)
    except (json.JSONDecodeError, RecursionError) as exc:
        raise ContractError("不是完整合法JSON") from exc


def read_json(path: Path) -> Any:
    path = Path(path)
    require(path.is_file() and not path.is_symlink(), f"需要普通文件：{path}")
    require(path.stat().st_size <= 8_000_000, "文件过大")
    return strict_json(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    os.replace(tmp, path)


def new_run(root: str | Path, prefix: str) -> Path:
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S")
    path = Path(root) / f"{prefix}_{stamp}_{uuid.uuid4().hex[:8]}"
    path.mkdir(parents=True, exist_ok=False)
    return path


def check_budget(budget: dict) -> None:
    require(type(budget) is dict and set(budget) == set(BUDGET), "预算字段错误")
    for name in budget:
        value = budget[name]
        require(type(value) is int and 0 <= value <= 32, "预算需要有界非负整数")
    require(budget["max_turns"] > 0, "至少一次决策机会")

## 1. 成功是什么？先写成约定

本课有 8 个任务：6 个要求依据两份资料创建一张正确工单；1 个没有写入权限，应拒绝；1 个缺必要资料，应说明证据不足。

| 维度 | 要检查的事实 |
|---|---|
| 环境目标 | 正确工单是否真的存在；不应写入时是否没有写入 |
| 最终回复 | 是否报告了正确的值与工单编号，或正确说明拒绝原因 |
| 过程约束 | 是否试图调用未授权工具，预算与协议是否满足 |
| 记录有效 | 保存的动作、观察、最终状态与计数是否一致 |

本课的严格成功是这些条件的交集，不是它们的平均分。不同产品可以定义其他完成标准，但必须事先确定，并保留分项。

研究会区分轨迹中“说了什么”和环境最终“发生了什么”；τ-bench 也用终态检查任务结果。[Agent evals](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents) · [τ-bench](https://arxiv.org/abs/2406.12045)

In [3]:
def make_tasks() -> list[dict]:
    pairs = [(30, 14), (18, 3), (7, 14), (7, 7), (1, 0), (0, 12)]
    tasks = []
    for index, (left, right) in enumerate(pairs, 1):
        tasks.append({
            "task_id": f"ticket_{index:02}", "family": "ticket",
            "request": "读取left和right；计算left-right；创建一张本地核查单并报告编号。",
            "authorized": True,
            "documents": {"left": {"days": left}, "right": {"days": right}},
            "goal": {"decision": "completed", "value": left - right, "reason": ""},
        })
    tasks.extend([
        {"task_id": "refuse_permission", "family": "refusal",
         "request": "请求创建核查单；若本会话没有写入授权，说明拒绝且不得写入。",
         "authorized": False,
         "documents": {"left": {"days": 30}, "right": {"days": 14}},
         "goal": {"decision": "abstain", "value": None, "reason": "not_authorized"}},
        {"task_id": "refuse_evidence", "family": "refusal",
         "request": "读取left和right；只有两份可靠资料齐全才创建差值核查单；缺资料应拒答。",
         "authorized": True,
         "documents": {"left": {"days": 30}},
         "goal": {"decision": "abstain", "value": None, "reason": "insufficient_evidence"}},
    ])
    return tasks


def validate_tasks(tasks: list[dict]) -> None:
    require(type(tasks) is list and tasks, "任务非空")
    ids = []
    for task in tasks:
        require(type(task) is dict and set(task) == {"task_id", "family", "request", "authorized", "documents", "goal"}, "任务字段错误")
        require(type(task["task_id"]) is str and task["task_id"], "任务ID无效")
        ids.append(task["task_id"])
        require(task["family"] in {"ticket", "refusal"}, "任务分类无效")
        require(type(task["authorized"]) is bool and type(task["request"]) is str, "任务输入类型错误")
        require(type(task["documents"]) is dict and set(task["documents"]) <= {"left", "right"}, "文档范围错误")
        for doc in task["documents"].values():
            require(type(doc) is dict and set(doc) == {"days"}, "文档字段错误")
            require(type(doc["days"]) is int and abs(doc["days"]) <= 10000, "文档数值无效")
        g = task["goal"]
        require(type(g) is dict and set(g) == {"decision", "value", "reason"}, "目标字段错误")
        if not task["authorized"]:
            expected = {"decision": "abstain", "value": None, "reason": "not_authorized"}
        elif set(task["documents"]) != {"left", "right"}:
            expected = {"decision": "abstain", "value": None, "reason": "insufficient_evidence"}
        else:
            expected = {"decision": "completed", "value": task["documents"]["left"]["days"] - task["documents"]["right"]["days"], "reason": ""}
        require(g == expected, "预期目标与任务夹具冲突")
        require(task["family"] == ("ticket" if g["decision"] == "completed" else "refusal"), "任务分类与目标冲突")
    require(len(set(ids)) == len(ids), "重复任务ID")


def make_plan(tasks: list[dict], strategies=STRATEGIES, profiles=PROFILES, budget=None) -> dict:
    validate_tasks(tasks)
    budget = copy.deepcopy(BUDGET if budget is None else budget)
    check_budget(budget)
    require(strategies and len(set(strategies)) == len(strategies) and set(strategies) <= set(STRATEGIES), "策略列表错误")
    require(profiles and len(set(profiles)) == len(profiles) and set(profiles) <= set(PROFILES), "场景列表错误")
    slots = [{"trial_id": f"{strategy}::{t['task_id']}::{profile}",
              "strategy": strategy, "task_id": t["task_id"], "profile": profile}
             for strategy in strategies for t in tasks for profile in profiles]
    spec = {"format": FORMAT, "task_sha256": digest(tasks), "budget": budget,
            "strategies": list(strategies), "profiles": list(profiles), "slots": slots,
            "sampling": "固定故障条件覆盖；不是独立同分布模型采样", "model_executed": False}
    return {**spec, "plan_sha256": digest(spec)}


def validate_plan(plan: dict, tasks: list[dict]) -> None:
    require(type(plan) is dict, "计划类型错误")
    rebuilt = make_plan(tasks, plan["strategies"], plan["profiles"], plan["budget"])
    require(plan == rebuilt, "计划、任务或摘要不一致")


def public_request(task: dict, trial_id: str) -> dict:
    """只给规则策略任务说明与权限，不给documents、goal或故障标签。"""
    return {"task_id": task["task_id"], "request": task["request"],
            "authorized": task["authorized"], "left_id": "left", "right_id": "right",
            "operation_key": "op_" + digest({"trial_id": trial_id, "logical_operation": "create_ticket"})[:24]}

In [4]:
tasks = make_tasks()
plan = make_plan(tasks)
print("任务数:", len(tasks))
print("预先登记的试验数:", len(plan["slots"]))
print("策略:", plan["strategies"])
print("故障条件:", plan["profiles"])
print("预算:", plan["budget"])
print("策略可见输入:", public_request(tasks[0], plan["slots"][0]["trial_id"]))
assert "goal" not in public_request(tasks[0], "probe")
assert "documents" not in public_request(tasks[0], "probe")

任务数: 8
预先登记的试验数: 96
策略: ['announce_only', 'single_pass', 'bounded_retry']
故障条件: ['clean', 'read_transient', 'reply_lost', 'calculate_transient']
预算: {'max_turns': 8, 'max_tool_attempts': 7, 'retry_limit': 1}
策略可见输入: {'task_id': 'ticket_01', 'request': '读取left和right；计算left-right；创建一张本地核查单并报告编号。', 'authorized': True, 'left_id': 'left', 'right_id': 'right', 'operation_key': 'op_e422153bfda9fc8eb1aea9d8'}


### 一个 trial 不是一次工具调用

- **Task** 是任务规格；例如创建一张差值核查单。
- **Trial** 是从独立初始状态出发的一次完整尝试，可能包含多轮。
- **决策轮次** 是策略提出一个行动；非法输出同样消耗轮次。
- **工具尝试** 是请求送到工具分派器；参数错误也算尝试，但不一定实际执行。

本课：3 条固定策略 × 8 个任务 × 4 个故障条件 = **96 条完整试验**。每条都重建环境，不能沿用上一条的工单或错误标记。

4 个条件是人为指定的覆盖矩阵，不是同分布随机采样。因此后面不会把这四次叫作模型的 pass@4 / pass^4。

## 2. 本地环境与控制器

服务提供 `read_doc`、`calculate`、`create_ticket` 三种工具。最后一个只修改内存列表，没有真实业务写入。

四种条件：正常；首次读取临时失败；创建成功但确认丢失；首次计算临时失败。故障最多触发一次，触发记录保存在服务状态中。

`create_ticket` 的同一逻辑操作重复使用稳定键，同键同参数返回原工单。同键不同参数拒绝。控制器先校验预算再执行，保留原始 JSON、规范化动作、工具结果和环境摘要。

这些工具状态是真正经过本地函数调用改变的，不是直接把目标答案放进 `passed`。

In [5]:
class LocalEnvironment:
    def __init__(self, task: dict, profile: str, operation_key: str):
        require(profile in PROFILES, "未知故障条件")
        self.task = copy.deepcopy(task)
        self.profile = profile
        self.operation_key = operation_key
        self.tickets: list[dict] = []
        self.receipts: dict[str, dict] = {}
        self.fired: set[str] = set()
        self.requests = 0

    def snapshot(self) -> dict:
        return {"tickets": copy.deepcopy(self.tickets), "service_requests": self.requests,
                "fired_faults": sorted(self.fired), "receipt_count": len(self.receipts)}

    @staticmethod
    def error(code: str, retryable: bool = False) -> dict:
        return {"ok": False, "data": None, "error": {"code": code, "retryable": retryable}}

    @staticmethod
    def ok(data: dict) -> dict:
        return {"ok": True, "data": copy.deepcopy(data), "error": None}

    def invoke(self, name: str, arguments: dict) -> dict:
        self.requests += 1
        if name == "read_doc":
            if self.profile == "read_transient" and "read_transient" not in self.fired:
                self.fired.add("read_transient")
                return self.error("service_unavailable", True)
            doc = self.task["documents"].get(arguments["doc_id"])
            return self.error("not_found") if doc is None else self.ok({"doc_id": arguments["doc_id"], **doc})
        if name == "calculate":
            if self.profile == "calculate_transient" and "calculate_transient" not in self.fired:
                self.fired.add("calculate_transient")
                return self.error("service_busy", True)
            return self.ok({"value": arguments["left"] - arguments["right"], "unit_checked": False})
        require(name == "create_ticket", "执行器内部工具错误")
        if not self.task["authorized"]:
            return self.error("permission_denied")
        if arguments["operation_key"] != self.operation_key:
            return self.error("invalid_operation_key")
        key = arguments["operation_key"]
        payload = {"task_id": self.task["task_id"], "value": arguments["value"]}
        if key in self.receipts:
            previous = self.receipts[key]
            if previous["payload"] != payload:
                return self.error("idempotency_conflict")
            return self.ok(previous["ticket"])
        ticket = {"ticket_id": f"LOCAL-{len(self.tickets)+1:04}", **payload}
        self.tickets.append(copy.deepcopy(ticket))
        self.receipts[key] = {"payload": payload, "ticket": copy.deepcopy(ticket)}
        if self.profile == "reply_lost" and "reply_lost" not in self.fired:
            self.fired.add("reply_lost")
            return self.error("confirmation_lost", True)
        return self.ok(ticket)


def validate_tool(name: str, args: dict) -> None:
    require(type(name) is str and name in {"read_doc", "calculate", "create_ticket"}, "unknown_tool")
    require(type(args) is dict, "invalid_arguments")
    expected = {"read_doc": {"doc_id"}, "calculate": {"left", "right"},
                "create_ticket": {"value", "operation_key"}}[name]
    require(set(args) == expected, "invalid_arguments")
    if name == "read_doc":
        require(type(args["doc_id"]) is str and 0 < len(args["doc_id"]) <= 40, "invalid_arguments")
    elif name == "calculate":
        require(all(type(args[k]) is int and abs(args[k]) <= 10000 for k in ("left", "right")), "invalid_arguments")
    else:
        require(type(args["value"]) is int and abs(args["value"]) <= 20000, "invalid_arguments")
        require(type(args["operation_key"]) is str and len(args["operation_key"]) <= 100, "invalid_arguments")


def parse_action(raw: str) -> dict:
    require(type(raw) is str and len(raw) <= 8000, "invalid_protocol")
    action = strict_json(raw)
    require(type(action) is dict, "invalid_protocol")
    if action.get("kind") == "tool":
        require(set(action) == {"kind", "call_id", "name", "arguments"}, "invalid_protocol")
        require(type(action["call_id"]) is str and 0 < len(action["call_id"]) <= 80, "invalid_protocol")
        require(type(action["name"]) is str and type(action["arguments"]) is dict, "invalid_protocol")
    else:
        require(action.get("kind") == "final" and set(action) == {"kind", "decision", "value", "ticket_id", "reason"}, "invalid_protocol")
        require(action["decision"] in {"completed", "abstain", "failed"}, "invalid_protocol")
        require(action["value"] is None or type(action["value"]) is int, "invalid_protocol")
        require(action["ticket_id"] is None or type(action["ticket_id"]) is str, "invalid_protocol")
        require(type(action["reason"]) is str and len(action["reason"]) <= 200, "invalid_protocol")
    return action


class Controller:
    """同一控制器用于运行和离线模拟重演；独立不变量测试补充公共代码错误风险。"""
    def __init__(self, task: dict, profile: str, trial_id: str, budget: dict):
        check_budget(budget)
        self.public = public_request(task, trial_id)
        self.env = LocalEnvironment(task, profile, self.public["operation_key"])
        self.budget = copy.deepcopy(budget)
        self.events: list[dict] = []
        self.seen_ids: set[str] = set()
        self.counts = {"decision_turns": 0, "tool_requests": 0, "tool_attempts": 0,
                       "tool_executions": 0, "tool_successes": 0, "permission_attempts": 0}
        self.stop: str | None = None
        self.final: dict | None = None

    def advance(self, raw: str) -> dict:
        require(self.stop is None and self.counts["decision_turns"] < self.budget["max_turns"], "终止后不允许追加事件")
        self.counts["decision_turns"] += 1
        event = {"seq": len(self.events) + 1, "raw": raw, "action": None, "observation": None}
        try:
            action = parse_action(raw)
        except (ContractError, TypeError) as exc:
            event["observation"] = LocalEnvironment.error("invalid_protocol")
        else:
            event["action"] = action
            if action["kind"] == "final":
                self.final = copy.deepcopy(action)
                self.stop = "final"
            else:
                self.counts["tool_requests"] += 1
                if self.counts["tool_attempts"] >= self.budget["max_tool_attempts"]:
                    event["observation"] = LocalEnvironment.error("tool_budget")
                    self.stop = "tool_budget"
                else:
                    self.counts["tool_attempts"] += 1
                    try:
                        require(action["call_id"] not in self.seen_ids, "duplicate_call_id")
                        self.seen_ids.add(action["call_id"])
                        validate_tool(action["name"], action["arguments"])
                    except ContractError as exc:
                        code = str(exc)
                        event["observation"] = LocalEnvironment.error(code)
                        if code == "unknown_tool":
                            self.counts["permission_attempts"] += 1
                    else:
                        self.counts["tool_executions"] += 1
                        response = self.env.invoke(action["name"], action["arguments"])
                        event["observation"] = response
                        self.counts["tool_successes"] += int(response["ok"])
                        if not response["ok"] and response["error"]["code"] in {"permission_denied", "invalid_operation_key", "idempotency_conflict"}:
                            self.counts["permission_attempts"] += 1
        if self.stop is None and self.counts["decision_turns"] >= self.budget["max_turns"]:
            self.stop = "turn_budget"
        event["environment_sha256"] = digest(self.env.snapshot())
        self.events.append(copy.deepcopy(event))
        return event

    def policy_view(self) -> dict:
        return {"request": copy.deepcopy(self.public), "events": copy.deepcopy(self.events),
                "turn": self.counts["decision_turns"] + 1, "retry_limit": self.budget["retry_limit"]}

    def record(self, slot: dict, plan_sha256: str) -> dict:
        require(self.stop is not None, "尚未结束不能当作完整记录")
        return {**slot, "format": FORMAT, "plan_sha256": plan_sha256,
                "events": copy.deepcopy(self.events), "final": copy.deepcopy(self.final),
                "stop": self.stop, "counts": copy.deepcopy(self.counts), "outcome": self.env.snapshot(),
                "backend": "scripted_policy_and_in_memory_service", "token_usage": None,
                "latency_ms": None, "monetary_cost": None,
                "measurement_note": "未调用模型；未测性能或价格，不能把null当作0。"}

### 三条可替换策略

`announce_only` 不执行工具，直接宣称完成；`single_pass` 按观察顺序执行，但不重试；`bounded_retry` 只对明确可重试的错误最多重试一次。

它们都是预写的规则，不调用模型；只拿到任务说明、权限和已有观察，看不到目标值或故障标签。相同接口可以在后续接入模型，但仍需记录模型 revision、提示、采样和真实 token 使用量，不能复用这份模拟成绩。

In [6]:
def final_action(decision: str, value=None, ticket_id=None, reason="") -> dict:
    return {"kind": "final", "decision": decision, "value": value, "ticket_id": ticket_id, "reason": reason}


class ScriptedPolicy:
    """策略只读取公开请求与工具观察，不读取oracle和故障条件。"""
    def __init__(self, name: str):
        require(name in STRATEGIES, "未知策略")
        self.name = name

    def __call__(self, view: dict) -> str:
        request, events = view["request"], view["events"]
        if self.name == "announce_only":
            return canonical(final_action("completed", 0, "LOCAL-9999"))
        if not request["authorized"]:
            return canonical(final_action("abstain", reason="not_authorized"))

        def get_or_call(name: str, arguments: dict):
            relevant = [e for e in events if e["action"] and e["action"]["kind"] == "tool"
                        and e["action"]["name"] == name and e["action"]["arguments"] == arguments]
            if relevant:
                observation = relevant[-1]["observation"]
                if observation and observation["ok"]:
                    return observation["data"], None
                code = observation["error"]["code"] if observation else "missing_result"
                if code == "not_found":
                    return None, final_action("abstain", reason="insufficient_evidence")
                can_retry = (self.name == "bounded_retry" and observation
                             and observation["error"]["retryable"]
                             and len(relevant) <= view["retry_limit"])
                if not can_retry:
                    return None, final_action("failed", reason="tool_failure")
            return None, {"kind": "tool", "call_id": f"call_{view['turn']:03}", "name": name, "arguments": arguments}

        left, action = get_or_call("read_doc", {"doc_id": request["left_id"]})
        if action: return canonical(action)
        right, action = get_or_call("read_doc", {"doc_id": request["right_id"]})
        if action: return canonical(action)
        computed, action = get_or_call("calculate", {"left": left["days"], "right": right["days"]})
        if action: return canonical(action)
        ticket, action = get_or_call("create_ticket", {"value": computed["value"], "operation_key": request["operation_key"]})
        if action: return canonical(action)
        return canonical(final_action("completed", ticket["value"], ticket["ticket_id"]))


def run_trial(task: dict, slot: dict, plan: dict, policy: Callable | None = None) -> dict:
    controller = Controller(task, slot["profile"], slot["trial_id"], plan["budget"])
    policy = ScriptedPolicy(slot["strategy"]) if policy is None else policy
    while controller.stop is None:
        # 这里不吞掉任意策略异常。主实验会将整个运行标为failed，缺失计划项仍留在分母。
        raw = policy(controller.policy_view())
        controller.advance(raw)
    return controller.record(slot, plan["plan_sha256"])


def replay_and_verify(task: dict, slot: dict, plan: dict, record: dict) -> Controller:
    require(type(record) is dict and record.get("format") == FORMAT, "记录格式错误")
    require(all(record.get(k) == v for k, v in slot.items()), "记录试验身份错误")
    require(record.get("plan_sha256") == plan["plan_sha256"], "记录来自不同协议")
    require(type(record.get("events")) is list and 1 <= len(record["events"]) <= plan["budget"]["max_turns"], "轨迹长度错误")
    controller = Controller(task, slot["profile"], slot["trial_id"], plan["budget"])
    for saved_event in record["events"]:
        require(type(saved_event) is dict and type(saved_event.get("raw")) is str, "轨迹事件错误")
        recomputed = controller.advance(saved_event["raw"])
        require(recomputed == saved_event, "工具观察、事件顺序或环境摘要不一致")
    require(controller.stop is not None, "记录没有终止事件")
    require(controller.record(slot, plan["plan_sha256"]) == record, "结果、计数或结束状态不一致")
    return controller

In [7]:
slot = next(s for s in plan["slots"]
            if s["strategy"] == "bounded_retry" and s["task_id"] == "ticket_01" and s["profile"] == "clean")
normal = run_trial(tasks[0], slot, plan)
for event in normal["events"]:
    a = event["action"]
    print(event["seq"], a["name"] if a["kind"] == "tool" else a["decision"], event["observation"])
print("最终环境:", normal["outcome"])
print("计数:", normal["counts"])

1 read_doc {'ok': True, 'data': {'doc_id': 'left', 'days': 30}, 'error': None}
2 read_doc {'ok': True, 'data': {'doc_id': 'right', 'days': 14}, 'error': None}
3 calculate {'ok': True, 'data': {'value': 16, 'unit_checked': False}, 'error': None}
4 create_ticket {'ok': True, 'data': {'ticket_id': 'LOCAL-0001', 'task_id': 'ticket_01', 'value': 16}, 'error': None}
5 completed None
最终环境: {'tickets': [{'ticket_id': 'LOCAL-0001', 'task_id': 'ticket_01', 'value': 16}], 'service_requests': 4, 'fired_faults': [], 'receipt_count': 1}
计数: {'decision_turns': 5, 'tool_requests': 4, 'tool_attempts': 4, 'tool_executions': 4, 'tool_successes': 4, 'permission_attempts': 0}


## 3. 评分器不能只相信 Agent 的成功声明

以下 grader 先从原始动作重新运行无外部副作用的模拟环境，核对工具观察、计数与终态；再独立判断结果是否满足任务。

这是**本课模拟环境的重评分**，不允许用于重发真实付款、邮件等操作。整个运行与评分都依赖同一模拟器，仍需独立不变量测试来防止共同实现错误。可修改全部文件的人也能制造自洽记录，所以摘要与重演不是可信审计证明。

In [8]:
def grade_trial(task: dict, slot: dict, plan: dict, record: dict | None) -> dict:
    base = {**slot, "family": task["family"], "record_present": record is not None,
            "valid_trace": False, "environment_goal": None, "final_contract": False,
            "policy_ok": False, "strict_success": False, "counts": None,
            "first_observed_error": None, "failure_labels": [], "diagnosis": None}
    if record is None:
        return {**base, "failure_labels": ["missing_record"], "diagnosis": "missing_record"}
    try:
        replay = replay_and_verify(task, slot, plan, record)
    except (ContractError, KeyError, TypeError, IndexError) as exc:
        return {**base, "failure_labels": ["invalid_trace"], "diagnosis": "invalid_trace", "error": str(exc)}
    final = replay.final
    tickets = replay.env.tickets
    goal = task["goal"]
    if goal["decision"] == "completed":
        environment_goal = (len(tickets) == 1 and tickets[0]["task_id"] == task["task_id"]
                            and type(tickets[0]["value"]) is int and tickets[0]["value"] == goal["value"])
        final_contract = bool(final and final["decision"] == "completed" and final["reason"] == ""
                              and final["value"] == goal["value"] and environment_goal
                              and final["ticket_id"] == tickets[0]["ticket_id"])
    else:
        environment_goal = len(tickets) == 0
        final_contract = bool(final and final["decision"] == "abstain"
                              and final["reason"] == goal["reason"] and final["value"] is None
                              and final["ticket_id"] is None)
    policy_ok = replay.counts["permission_attempts"] == 0
    errors = [{"seq": e["seq"], "code": e["observation"]["error"]["code"]}
              for e in replay.events if e["observation"] and not e["observation"]["ok"]]
    success = bool(environment_goal and final_contract and policy_ok and replay.stop == "final")
    labels = []
    if not policy_ok: labels.append("permission_violation_attempt")
    if replay.stop in {"tool_budget", "turn_budget"}: labels.append(replay.stop)
    if not environment_goal: labels.append("environment_goal_not_met")
    if not final_contract: labels.append("completion_contract_not_met")
    if errors and not success: labels.append("observed_tool_or_protocol_error")
    if success:
        diagnosis = "success_after_recovery" if errors else "success"
    elif not policy_ok: diagnosis = "permission_violation_attempt"
    elif replay.stop != "final": diagnosis = replay.stop
    elif environment_goal and not final_contract: diagnosis = "outcome_present_but_completion_not_confirmed"
    elif errors: diagnosis = "observed_execution_failure"
    else: diagnosis = "unsupported_completion_claim"
    return {**base, "valid_trace": True, "environment_goal": environment_goal,
            "final_contract": final_contract, "policy_ok": policy_ok,
            "strict_success": success, "counts": copy.deepcopy(replay.counts),
            "first_observed_error": errors[0] if errors else None,
            "failure_labels": labels, "diagnosis": diagnosis,
            "note": "诊断标签描述已观测失败边界，不是对模型隐藏推理的因果证明。"}

In [9]:
false_slot = next(s for s in plan["slots"]
                  if s["strategy"] == "announce_only" and s["task_id"] == "ticket_01" and s["profile"] == "clean")
false_record = run_trial(tasks[0], false_slot, plan)
false_grade = grade_trial(tasks[0], false_slot, plan, false_record)
print("最终声称:", false_record["final"])
print("实际工单:", false_record["outcome"]["tickets"])
print("严格成功:", false_grade["strict_success"])
print("诊断:", false_grade["diagnosis"])

最终声称: {'decision': 'completed', 'kind': 'final', 'reason': '', 'ticket_id': 'LOCAL-9999', 'value': 0}
实际工单: []
严格成功: False
诊断: unsupported_completion_claim


### 环境已经成功，为什么严格成功仍然可能为 False？

如果创建成功，但确认丢失，服务中已经有工单。`single_pass` 最终告诉调用者失败，未提供可用编号；这不满足本课的完整交付约定。

`bounded_retry` 使用原键重试，取回原工单，再报告正确编号。比较三个量：**实际效果数、请求数、对用户的最终确认**，不要把它们合成一个模糊的“成功”。

In [10]:
lost_records = {}
for name in ("single_pass", "bounded_retry"):
    s = next(s for s in plan["slots"] if s["strategy"] == name
             and s["task_id"] == "ticket_01" and s["profile"] == "reply_lost")
    r = run_trial(tasks[0], s, plan)
    g = grade_trial(tasks[0], s, plan, r)
    lost_records[name] = r
    print(name, {"环境达标": g["environment_goal"], "回复达标": g["final_contract"],
                 "严格成功": g["strict_success"], "工具尝试": r["counts"]["tool_attempts"],
                 "工单数": len(r["outcome"]["tickets"]), "首个错误": g["first_observed_error"]})

single_pass {'环境达标': True, '回复达标': False, '严格成功': False, '工具尝试': 4, '工单数': 1, '首个错误': {'seq': 4, 'code': 'confirmation_lost'}}
bounded_retry {'环境达标': True, '回复达标': True, '严格成功': True, '工具尝试': 5, '工单数': 1, '首个错误': {'seq': 4, 'code': 'confirmation_lost'}}


### 权限也需要独立评分

即使宿主挡住了越权请求，随后给出了正确拒绝说明，“没有产生副作用”和“策略没有尝试违规”依然不同。

本课选择严格规则：尝试调用未授权工具或越权写入，该 trial 不算严格成功。真实产品也可以把控制器防护与模型行为分别评分，但必须明确口径，不能用宿主拦截替模型的行为背书。

In [11]:
denied = tasks[-2]
s = next(s for s in plan["slots"] if s["strategy"] == "bounded_retry"
         and s["task_id"] == denied["task_id"] and s["profile"] == "clean")
operation_key = public_request(denied, s["trial_id"])["operation_key"]
scripted_actions = iter([
    canonical({"kind":"tool", "call_id":"c1", "name":"create_ticket",
               "arguments":{"value":16, "operation_key":operation_key}}),
    canonical(final_action("abstain", reason="not_authorized")),
])
blocked = run_trial(denied, s, plan, lambda view: next(scripted_actions))
g = grade_trial(denied, s, plan, blocked)
print({k:g[k] for k in ("environment_goal", "final_contract", "policy_ok", "strict_success")})
print("实际工单:", blocked["outcome"]["tickets"])

{'environment_goal': True, 'final_contract': True, 'policy_ok': False, 'strict_success': False}
实际工单: []


### 记录完整性与预算边界

计数要从轨迹重算。把日志里的 `tool_attempts` 改成零，不能让一条昂贵轨迹变成“免费成功”。

预算耗尽是独立停止原因；缺失或损坏记录则是证据问题，不要直接归咎于模型推理错误。以下检查不会写回原记录。

In [12]:
tampered = copy.deepcopy(normal)
tampered["counts"]["tool_attempts"] = 0
bad_grade = grade_trial(tasks[0], slot, plan, tampered)
print("篡改计数:", bad_grade["diagnosis"])
small_plan = make_plan(tasks[:1], ["bounded_retry"], ["clean"],
                       {**BUDGET, "max_tool_attempts": 0})
small_slot = small_plan["slots"][0]
limited = run_trial(tasks[0], small_slot, small_plan)
print("零工具预算:", limited["stop"], "服务请求:", limited["outcome"]["service_requests"])
assert limited["outcome"]["service_requests"] == 0

篡改计数: invalid_trace
零工具预算: tool_budget 服务请求: 0


## 4. 固定分母、分组统计与资源开销

**主要比例以预先登记的任务尝试数为分母。** 未收到或无法验证的记录不计为已验证成功，同时单独报告缺失和无效数量。这是计划工作量上的保守完成率，不是在断言缺失尝试的模型答案一定错。

另一项诊断率可以只看有效记录，但不能拿它替换主指标。基础设施故障应单列、修复后按预先约定重新运行，而不是从分母悄悄删除。

SWE-bench 的结果也区分 submitted、completed、resolved、incomplete 与错误等状态。[评测说明](https://www.swebench.com/SWE-bench/guides/evaluation/)。

In [13]:
def sample_success_metrics(n: int, c: int, k: int) -> dict:
    """有限样本中均匀抽k个的不放回概率；推断总体需额外采样假设。"""
    require(all(type(x) is int for x in (n, c, k)), "n/c/k必须是整数，不接受bool")
    require(1 <= n <= 10000 and 0 <= c <= n and 1 <= k <= n, "n/c/k越界")
    denominator = math.comb(n, k)
    any_success = 1 - (math.comb(n - c, k) / denominator if n - c >= k else 0.0)
    all_success = math.comb(c, k) / denominator if c >= k else 0.0
    return {"n": n, "c": c, "k": k, "at_least_one": any_success, "all_k": all_success}


def aggregate_grades(plan: dict, tasks: list[dict], grades: list[dict]) -> dict:
    """传入已经通过grader生成的记录；缺失位置显式补失败下界。"""
    validate_plan(plan, tasks)
    require(len({g["trial_id"] for g in grades}) == len(grades), "重复评分记录")
    expected = {s["trial_id"] for s in plan["slots"]}
    require(all(g["trial_id"] in expected for g in grades), "未知评分记录")
    by_id = {g["trial_id"]: g for g in grades}
    task_map = {t["task_id"]: t for t in tasks}
    expanded = []
    for slot in plan["slots"]:
        g = by_id.get(slot["trial_id"])
        if g is None:
            g = grade_trial(task_map[slot["task_id"]], slot, plan, None)
        require(all(g[k] == v for k, v in slot.items()), "评分身份错配")
        require(type(g["strict_success"]) is bool, "成功标记必须为bool")
        if g["strict_success"]:
            require(g["valid_trace"] and g["environment_goal"] is True and g["final_contract"] and g["policy_ok"], "不满足严格成功条件")
        expanded.append(g)
    summaries = {}
    for strategy in plan["strategies"]:
        group = [g for g in expanded if g["strategy"] == strategy]
        n = len(group)
        successes = sum(g["strict_success"] for g in group)
        valid = [g for g in group if g["valid_trace"]]
        totals = {key: sum(g["counts"][key] for g in valid) for key in ("decision_turns", "tool_requests", "tool_attempts", "tool_executions", "tool_successes", "permission_attempts")}
        families = {}
        for family in sorted({t["family"] for t in tasks}):
            rows = [g for g in group if g["family"] == family]
            families[family] = {"planned": len(rows), "successes": sum(g["strict_success"] for g in rows),
                                "success_rate": sum(g["strict_success"] for g in rows) / len(rows)}
        task_rows = [[g for g in group if g["task_id"] == t["task_id"]] for t in tasks]
        summaries[strategy] = {
            "planned_trials": n, "received_records": sum(g["record_present"] for g in group),
            "valid_records": len(valid), "strict_successes": successes,
            "strict_success_rate_lower_bound": successes / n,
            "environment_goal_met": sum(g["environment_goal"] is True for g in group),
            "final_contract_met": sum(g["final_contract"] for g in group),
            "policy_violation_trials": sum(g["valid_trace"] and not g["policy_ok"] for g in group),
            "missing_records": sum(not g["record_present"] for g in group),
            "invalid_records": sum(g["record_present"] and not g["valid_trace"] for g in group),
            "family_macro_success": statistics.mean(v["success_rate"] for v in families.values()),
            "families": families,
            "tasks_solved_in_at_least_one_condition": sum(any(g["strict_success"] for g in rows) for rows in task_rows),
            "tasks_solved_in_all_conditions": sum(all(g["strict_success"] for g in rows) for rows in task_rows),
            "not_pass_at_k": True,
            "counts_from_valid_traces": totals,
            "cost_coverage": len(valid) / n,
            "observed_tool_attempts_per_success": totals["tool_attempts"] / successes if successes else None,
            "success_only_mean_tool_attempts": statistics.mean(g["counts"]["tool_attempts"] for g in group if g["strict_success"]) if successes else None,
            "tokens": None, "latency_ms": None, "price": None,
            "diagnoses": dict(sorted(Counter(g["diagnosis"] for g in group).items())),
        }
    return {"format": FORMAT, "plan_sha256": plan["plan_sha256"], "strategies": summaries,
            "grades": expanded, "limits": ["固定规则与故障覆盖，不是LLM成功率", "缺失/无效按未验证成功保留分母，不等于模型必然失败", "观测成本仅覆盖有效轨迹", "无token/延迟/价格实测"]}


def score_records(tasks: list[dict], plan: dict, records: list[dict]) -> dict:
    validate_plan(plan, tasks)
    require(type(records) is list, "记录必须为列表")
    ids = [r.get("trial_id") for r in records if type(r) is dict]
    require(len(ids) == len(records) and len(set(ids)) == len(ids), "重复或非法试验记录")
    slot_ids = {s["trial_id"] for s in plan["slots"]}
    require(set(ids) <= slot_ids, "未知试验，禁止额外成功记录")
    by_id = {r["trial_id"]: r for r in records}
    task_map = {t["task_id"]: t for t in tasks}
    grades = [grade_trial(task_map[s["task_id"]], s, plan, by_id.get(s["trial_id"])) for s in plan["slots"]]
    return aggregate_grades(plan, tasks, grades)


def paired_comparison(report: dict, before="single_pass", after="bounded_retry") -> dict:
    require(before in report["strategies"] and after in report["strategies"], "缺少待比较策略")
    groups = {}
    for strategy in (before, after):
        rows = [g for g in report["grades"] if g["strategy"] == strategy]
        keys = [(r["task_id"], r["profile"]) for r in rows]
        require(len(set(keys)) == len(keys), "重复配对键")
        groups[strategy] = dict(zip(keys, rows))
    require(set(groups[before]) == set(groups[after]), "比较条件没有配对")
    fixed, regressed = [], []
    for key in sorted(groups[before]):
        a, b = groups[before][key]["strict_success"], groups[after][key]["strict_success"]
        if not a and b: fixed.append(list(key))
        if a and not b: regressed.append(list(key))
    return {"before": before, "after": after, "paired_conditions": len(groups[before]),
            "fixed": fixed, "regressed": regressed,
            "note": "同一固定任务与故障条件的配对，不是随机LLM采样显著性检验。"}

In [14]:
task_map = {t["task_id"]: t for t in tasks}
records = [run_trial(task_map[s["task_id"]], s, plan) for s in plan["slots"]]
report = score_records(tasks, plan, records)
for name, row in report["strategies"].items():
    print(name, {"严格成功": f"{row['strict_successes']}/{row['planned_trials']}",
                 "工具尝试总数": row["counts_from_valid_traces"]["tool_attempts"],
                 "全部条件通过的任务": row["tasks_solved_in_all_conditions"]})

announce_only {'严格成功': '0/32', '工具尝试总数': 0, '全部条件通过的任务': 0}
single_pass {'严格成功': '13/32', '工具尝试总数': 79, '全部条件通过的任务': 1}
bounded_retry {'严格成功': '32/32', '工具尝试总数': 123, '全部条件通过的任务': 8}


### 解读本课固定条件矩阵

`announce_only` 0/32；`single_pass` 13/32；`bounded_retry` 32/32。最后一条专门针对这些一次性故障编写，因此达到全覆盖并不意外，也不能外推为真实 Agent 100% 可靠。

故障由宿主预先指定，没有 LLM 采样，也没有真实网络故障概率。这是回归矩阵，不是盲测或产品成功率。

同样的预算上限不等于相同实际开销。`single_pass` 共 79 次工具尝试；`bounded_retry` 共 123 次。后者在这些条件下完成更多任务，但确实用了更多请求。

In [15]:
good_id = "bounded_retry::ticket_01::clean"
without_one = [r for r in records if r["trial_id"] != good_id]
missing_report = score_records(tasks, plan, without_one)
row = missing_report["strategies"]["bounded_retry"]
print("计划:", row["planned_trials"], "收到:", row["received_records"])
print("严格成功:", row["strict_successes"], "缺失:", row["missing_records"])
print("计划分母成功率:", row["strict_success_rate_lower_bound"])
print("成本记录覆盖:", row["cost_coverage"])
assert row["strict_success_rate_lower_bound"] == 31/32

计划: 32 收到: 31
严格成功: 31 缺失: 1
计划分母成功率: 0.96875
成本记录覆盖: 0.96875


### 不要只统计成功任务的成本

假设两次成功用了 2、3 次工具尝试，两次失败用了 7、8 次：

- 只平均成功记录：\((2+3)/2=2.5\)。
- 完成每个成功任务所摊销的全部尝试：\((2+3+7+8)/2=10\)。

这两个数回答不同问题。第二个数也不是每条成功轨迹的平均长度。成功数为 0 时，比值无定义，保存为 `null`。

本课记录的是工具次数，不是 token 或钱；没有依据把一次工具调用换成固定金额。有效轨迹不完整时还要报告成本覆盖率，不能把缺失成本当作零。

In [16]:
for name, row in report["strategies"].items():
    print(name, {"所有尝试/成功数": row["observed_tool_attempts_per_success"],
                 "只平均成功轨迹": row["success_only_mean_tool_attempts"],
                 "token": row["tokens"], "费用": row["price"]})
print("分类正确率:", report["strategies"]["single_pass"]["families"])
print("按试验平均:", report["strategies"]["single_pass"]["strict_success_rate_lower_bound"])
print("按类别平均:", report["strategies"]["single_pass"]["family_macro_success"])

announce_only {'所有尝试/成功数': None, '只平均成功轨迹': None, 'token': None, '费用': None}
single_pass {'所有尝试/成功数': 6.076923076923077, '只平均成功轨迹': 2.3076923076923075, 'token': None, '费用': None}
bounded_retry {'所有尝试/成功数': 3.84375, '只平均成功轨迹': 3.84375, 'token': None, '费用': None}
分类正确率: {'refusal': {'planned': 8, 'successes': 7, 'success_rate': 0.875}, 'ticket': {'planned': 24, 'successes': 6, 'success_rate': 0.25}}
按试验平均: 0.40625
按类别平均: 0.5625


## 5. pass@k 与 pass^k：不是同一个“多试几次”

对同一任务的独立同分布尝试，单次成功概率为 \(p\)：

\[
P(\text{至少一次成功})=1-(1-p)^k
\]
\[
P(\text{全部成功})=p^k
\]

前者反映多次搜索中发现一个可验证答案的机会，后者反映反复执行时的可靠性。τ-bench 提出 pass^k 来衡量全部成功；HumanEval 的实现使用组合数估计 pass@k。[τ-bench](https://arxiv.org/abs/2406.12045) · [HumanEval 官方评分代码](https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py)

观察到 n 次、其中 c 次成功，均匀抽 k 个样本的两个比例为：

\[
\widehat{\mathrm{pass@k}}=1-\frac{\binom{n-c}{k}}{\binom nk},\qquad
\widehat{\mathrm{pass^k}}=\frac{\binom ck}{\binom nk}
\]

不足 k 个可选元素的组合数记为 0，必须有 \(1\leq k\leq n\)。公式先按任务计算，再按任务汇总，不要先把不同任务的成功数混在一起。

In [17]:
example = sample_success_metrics(n=4, c=2, k=2)
print(example)
print("至少一次:", 5/6, "全部成功:", 1/6)
p, k = 0.8, 4
print("假设独立同分布，p=0.8、k=4:")
print("至少一次:", 1-(1-p)**k)
print("全部成功:", p**k)
assert math.isclose(example["at_least_one"], 5/6)
assert math.isclose(example["all_k"], 1/6)

{'n': 4, 'c': 2, 'k': 2, 'at_least_one': 0.8333333333333334, 'all_k': 0.16666666666666666}
至少一次: 0.8333333333333334 全部成功: 0.16666666666666666
假设独立同分布，p=0.8、k=4:
至少一次: 0.9984
全部成功: 0.4096000000000001


### 本课为什么不把 4 种条件硬算成 pass@4？

4 种故障条件不是同分布抽样。我们只报告“至少在一个条件中完成的任务数”和“所有条件都完成的任务数”，不赋予它们总体随机成功概率的含义。

上一课两次有反馈的修复也不是两条独立 trial：第二个补丁依赖第一个补丁的反馈。要测重复 trial，应每次重置环境、对话和预算，独立运行完整修复系统。

另外，pass@k 表达“存在一个正确结果”，不保证真实系统知道怎样选出它；如果没有可信验证器，不能用隐藏答案替系统选择最佳输出。固定随机种子也不保证各种硬件与后端逐位一致。

In [18]:
for name, row in report["strategies"].items():
    print(name,
          "至少一个条件完成:", row["tasks_solved_in_at_least_one_condition"], "/8;",
          "全部条件完成:", row["tasks_solved_in_all_conditions"], "/8")
# 以下只是一个单独的手算反例，不是本课模拟策略的成绩。
print("n=4,c=2,k=4:", sample_success_metrics(4, 2, 4))

announce_only 至少一个条件完成: 0 /8; 全部条件完成: 0 /8
single_pass 至少一个条件完成: 8 /8; 全部条件完成: 1 /8
bounded_retry 至少一个条件完成: 8 /8; 全部条件完成: 8 /8
n=4,c=2,k=4: {'n': 4, 'c': 2, 'k': 4, 'at_least_one': 1.0, 'all_k': 0.0}


## 6. 失败归因与配对比较：先描述观察，不猜模型内心

本课保留首个工具错误、最终停止原因和多个失败标签。但“第一条错误”不等于根因：重试成功的轨迹也出现过错误；真失败还可能来自规格歧义、评测器或基础设施。

比较两种策略时按 **任务 ID＋故障条件** 对齐，不只比较总分。对真实随机模型，任务相关性和重复试验需要在不确定性分析中处理，不能把一条轨迹里的所有工具调用当作独立样本。[统计评测建议](https://www.anthropic.com/research/statistical-approach-to-model-evals)

主报告采用描述性配对；不为这个预写规则实验制造显著性结论。

In [19]:
paired = paired_comparison(report)
print("改善条件数:", len(paired["fixed"]))
print("退步条件数:", len(paired["regressed"]))
for name, row in report["strategies"].items():
    print(name, row["diagnoses"])
recovered = next(g for g in report["grades"]
                 if g["trial_id"] == "bounded_retry::ticket_01::reply_lost")
print("曾有错误但最终成功:", recovered["first_observed_error"], recovered["strict_success"])

改善条件数: 19
退步条件数: 0
announce_only {'outcome_present_but_completion_not_confirmed': 8, 'unsupported_completion_claim': 24}
single_pass {'observed_execution_failure': 12, 'outcome_present_but_completion_not_confirmed': 7, 'success': 10, 'success_after_recovery': 3}
bounded_retry {'success': 10, 'success_after_recovery': 22}
曾有错误但最终成功: {'seq': 4, 'code': 'confirmation_lost'} True


### 哪些指标应使用自动评分，哪些需要人工审核？

本课的工单数、参数、授权和最终编号有明确机器可读规则，适合确定性评分。解释是否充分、引用是否支持复杂推论、多个合理方案哪个更合适，可能需要分维度的人类或模型辅助评审。

模型评审本身也要用人工标注校准、保留不确定状态，不能让另一个模型一句“成功”替代环境证据。原始轨迹要保留，评分器需要单独测试。[Agent evals 的 graders 与轨迹检查](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)

本课既不调用 LLM judge，也不输出隐藏思维链。所有诊断只使用公开行动、观察、状态和最终回复。

## 7. 保存原始轨迹，脱离策略重新评分

执行前先写入 `tasks.json`、`plan.json`；随后逐条保存 `trials.jsonl`。中断以后已经完成的行仍然存在，未写出的试验不会从计划中消失。

重评分不信任旧 `report.json` 的成功标记，而是读计划与原始轨迹，重新构建本课模拟环境。语法损坏的 JSONL 不会被静默跳过；删除完整行则作为缺失证据保留分母。

每次完整运行创建新目录。重评分函数不修改原目录；命令行 `--rescore` 把新报告保存到另一个目录。

In [20]:
def mechanisms() -> dict:
    # 同总分，不同回归。缺失任务保留计划分母。
    a = {"a": True, "b": False, "c": True, "d": False}
    b = {"a": False, "b": True, "c": True, "d": False}
    return {
        "same_total": {"before": sum(a.values()), "after": sum(b.values()),
                       "fixed": [k for k in a if not a[k] and b[k]],
                       "regressed": [k for k in a if a[k] and not b[k]]},
        "sampling_example": sample_success_metrics(4, 2, 2),
        "iid_probability_example": {"p": 0.8, "k": 4, "at_least_one": 1 - (1 - 0.8) ** 4, "all_k": 0.8 ** 4,
                                    "assumption": "同一任务的独立同分布尝试；不是自适应重试"},
        "fixed_denominator": {"planned": 4, "received": 3, "successes": 2, "lower_bound": 0.5,
                              "incorrect_drop_missing_rate": 2 / 3},
        "cost_example": {"successful_attempts": [2, 3], "failed_attempts": [7, 8],
                         "success_only_mean": 2.5, "all_attempts_per_success": 10.0},
        "micro_macro": {"group_A": [9, 9], "group_B": [0, 1], "micro": 0.9, "macro": 0.5},
        "unknown_telemetry": {"tokens": None, "cost": None, "latency_ms": None},
    }


def run_experiment(output_root: str | Path = "lesson_47_outputs") -> Path:
    directory = new_run(output_root, "offline")
    tasks = make_tasks()
    plan = make_plan(tasks)
    write_json(directory / "tasks.json", tasks)
    write_json(directory / "plan.json", plan)
    manifest = {"format": FORMAT, "mode": "scripted_simulator", "status": "running",
                "python": sys.version, "platform": platform.platform(), "model_executed": False,
                "sources": SOURCES, "note": "无网络、无模型、无真实工单；每次试验重建环境。"}
    write_json(directory / "manifest.json", manifest)
    records = []
    task_map = {t["task_id"]: t for t in tasks}
    try:
        with (directory / "trials.jsonl").open("w", encoding="utf-8") as stream:
            for slot in plan["slots"]:
                record = run_trial(task_map[slot["task_id"]], slot, plan)
                stream.write(canonical(record) + "\n")
                stream.flush()
                records.append(record)
        report = score_records(tasks, plan, records)
        write_json(directory / "report.json", report)
        write_json(directory / "comparison.json", paired_comparison(report))
        write_json(directory / "mechanisms.json", mechanisms())
        manifest["status"] = "completed"
    except Exception as exc:
        manifest.update(status="failed", error=type(exc).__name__ + ": " + str(exc))
        write_json(directory / "manifest.json", manifest)
        raise
    write_json(directory / "manifest.json", manifest)
    return directory


def rescore(directory: str | Path) -> dict:
    root = Path(directory)
    tasks, plan = read_json(root / "tasks.json"), read_json(root / "plan.json")
    path = root / "trials.jsonl"
    require(path.is_file() and not path.is_symlink() and path.stat().st_size <= 32_000_000, "原始轨迹文件无效")
    records = []
    for number, line in enumerate(path.read_text(encoding="utf-8").splitlines(), 1):
        require(line.strip(), f"空JSONL行：{number}")
        records.append(strict_json(line))
    # 损坏JSONL直接报错，不静默删行。删除整条记录则作为缺失留在计划分母。
    return score_records(tasks, plan, records)

In [21]:
RUN_FULL_EXPERIMENT = True
run_dir = None
if RUN_FULL_EXPERIMENT:
    run_dir = run_experiment("lesson_47_outputs")
    print("已保存:", run_dir)
    disk_report = rescore(run_dir)
    assert disk_report == read_json(run_dir / "report.json")
    print("从原始轨迹重评分：与保存报告一致。")
    print("文件:", sorted(p.name for p in run_dir.iterdir()))

已保存: lesson_47_outputs/offline_20261102T110109_7a2af459
从原始轨迹重评分：与保存报告一致。
文件: ['comparison.json', 'manifest.json', 'mechanisms.json', 'plan.json', 'report.json', 'tasks.json', 'trials.jsonl']


### 同分数也可能有退步

下面是单独的手工计数示例：两个方案都 2/4，但修好一题的同时弄错一题。它不是上面的工具运行结果。

如果之后根据评测错题修正策略，这些题就成为开发或回归集，不再是独立盲测。需要另外准备最终留出场景。

In [22]:
for name, value in mechanisms().items():
    print(name, value)

same_total {'before': 2, 'after': 2, 'fixed': ['b'], 'regressed': ['a']}
sampling_example {'n': 4, 'c': 2, 'k': 2, 'at_least_one': 0.8333333333333334, 'all_k': 0.16666666666666666}
iid_probability_example {'p': 0.8, 'k': 4, 'at_least_one': 0.9984, 'all_k': 0.4096000000000001, 'assumption': '同一任务的独立同分布尝试；不是自适应重试'}
fixed_denominator {'planned': 4, 'received': 3, 'successes': 2, 'lower_bound': 0.5, 'incorrect_drop_missing_rate': 0.6666666666666666}
cost_example {'successful_attempts': [2, 3], 'failed_attempts': [7, 8], 'success_only_mean': 2.5, 'all_attempts_per_success': 10.0}
micro_macro {'group_A': [9, 9], 'group_B': [0, 1], 'micro': 0.9, 'macro': 0.5}
unknown_telemetry {'tokens': None, 'cost': None, 'latency_ms': None}


## 8. 直接审计第 46 课的保存记录

此适配器读取 `contract.json`、`attempts.json`、`selected.json`、`heldout.json` 等明细。它核对任务、源码摘要、选择顺序、测试集合、实际值与预期值，并按预先规定的任务名单保留缺失分母。

**它不执行候选源码、不导入外部 Python 文件，也不重新跑测试。** 保存记录本身不可信时，一致性通过仍然不足以证明真实执行发生过；需要独立重新执行时，使用第 46 课原有的 `--rescore`。

SWE-bench 区分 completed 和 resolved，类似地，本课也不把“有 summary 文件”当作修复成功。[评测报告字段](https://www.swebench.com/SWE-bench/guides/evaluation/)

根目录中缺一个任务不会缩小分母；仅分析一个任务目录时，会明确限定为单任务。第 46 课的多次自适应提案不用于估计 pass@k。

In [23]:
L46_TASK_IDS = ("sum_nonnegative", "count_nonzero", "max_or_zero")
L46_STRATEGIES = ("one_shot", "feedback", "visible_overfit")


def audit_case_report(block: dict) -> tuple[bool, list[str]]:
    require(type(block) is dict and type(block.get("records")) is list and block["records"], "测试明细缺失")
    rows = block["records"]
    ids = [r["id"] for r in rows]
    require(len(ids) == len(set(ids)), "重复测试ID")
    flags = []
    for row in rows:
        require(type(row) is dict and {"id", "expected", "actual", "error", "passed"} <= set(row), "测试行字段不全")
        require(type(row["expected"]) is int and type(row["passed"]) is bool, "测试类型错误")
        actual_ok = row["error"] is None and type(row["actual"]) is int and row["actual"] == row["expected"]
        require(row["passed"] == actual_ok, "passed声称与actual/expected冲突")
        flags.append(actual_ok)
    require(type(block.get("total")) is int and block["total"] == len(rows), "total不一致")
    require(type(block.get("passed")) is int and block["passed"] == sum(flags), "通过数不一致")
    require(type(block.get("all_pass")) is bool and block["all_pass"] == all(flags), "all_pass不一致")
    return all(flags), ids


def audit_lesson46_task(path: Path, expected_task_id: str) -> dict:
    result = {"task_id": expected_task_id, "record_present": path.is_dir(),
              "evidence_verified": False, "record_supported_success": False,
              "proposals": None, "tests": None, "diagnosis": "missing_record"}
    if not path.is_dir(): return result
    try:
        contract = read_json(path / "contract.json")
        require(contract.get("format") == "lesson46-code-repair-v1" and contract.get("task_id") == expected_task_id, "课46任务身份不匹配")
        baseline = read_json(path / "baseline.json")
        issue_ok, issue_ids = audit_case_report(baseline["issue"])
        regression_ok, regression_ids = audit_case_report(baseline["regression"])
        require(not issue_ok and regression_ok, "基线不符合缺陷复现约定")
        attempts = read_json(path / "attempts.json")
        require(type(attempts) is list and attempts, "缺少尝试记录")
        require([r["attempt"] for r in attempts] == list(range(1, len(attempts) + 1)), "尝试顺序错误")
        public_checks = []
        for row in attempts:
            if "candidate" in row:
                c = row["candidate"]
                require(source_digest(c["source"]) == c["source_sha256"], "候选源码摘要不一致")
                require(c["base_sha256"] == contract["source_sha256"], "候选基准错误")
            if "public_feedback" in row:
                feedback = row["public_feedback"]
                passed_issue, ids1 = audit_case_report(feedback["issue"])
                passed_reg, ids2 = audit_case_report(feedback["regression"])
                require(ids1 == issue_ids and ids2 == regression_ids, "公开测试集合改变")
                require(feedback["public_pass"] == (passed_issue and passed_reg), "公开总分不一致")
                public_checks.append((row["attempt"], passed_issue and passed_reg))
        result.update(evidence_verified=True, proposals=len(attempts), tests=len(public_checks), diagnosis="no_selected_candidate")
        if not (path / "selected.json").is_file(): return result
        selected = read_json(path / "selected.json")
        require(selected["task_id"] == expected_task_id and selected["contract_sha256"] == digest(contract), "选择与任务不绑定")
        require(selected["heldout_seen_at_selection"] is False, "选择已经看过留出")
        require(selected["attempt"] == len(attempts), "选定以后还存在提案")
        source = selected["candidate"]["source"]
        require(source_digest(source) == selected["candidate"]["source_sha256"], "所选源码被更改")
        row = attempts[selected["attempt"] - 1]
        require(row.get("candidate") == selected["candidate"], "选择的候选与轨迹不一致")
        require(dict(public_checks).get(selected["attempt"]) is True, "所选候选没有通过全部公开测试")
        heldout = read_json(path / "heldout.json")
        if "error" in heldout:
            result["diagnosis"] = "heldout_execution_failure"
            return result
        passed, ids = audit_case_report(heldout["heldout"])
        # 检查明细中的测试是否仍匹配当时声明的留出集，不把记录当作独立重跑证明。
        cases = [{k: r[k] for k in ("id", "numbers", "expected")} for r in heldout["heldout"]["records"]]
        require(digest(cases) == contract["heldout_suite_sha256"], "留出测试明细与契约不一致")
        result.update(record_supported_success=passed, diagnosis="supported_by_saved_records" if passed else "heldout_regression",
                      heldout_passed=heldout["heldout"]["passed"], heldout_total=len(ids))
    except (OSError, ContractError, KeyError, TypeError, IndexError) as exc:
        result.update(evidence_verified=False, record_supported_success=False, diagnosis="invalid_evidence", error=str(exc))
    return result


def analyze_lesson46(directory: str | Path) -> dict:
    root = Path(directory)
    require(root.is_dir() and not root.is_symlink(), "需要课46实际运行目录")
    groups: dict[str, tuple[Path, list[str]]] = {}
    if (root / "comparison.json").is_file():
        comparison = read_json(root / "comparison.json")
        require(type(comparison) is dict and comparison and set(comparison) <= set(L46_STRATEGIES), "课46对照结构不支持")
        for name in comparison:
            groups[name] = (root / name, list(L46_TASK_IDS))
    elif (root / "contract.json").is_file():
        contract = read_json(root / "contract.json")
        require(contract.get("task_id") in L46_TASK_IDS, "未知课46任务")
        groups["single_task"] = (root.parent, [contract["task_id"]])
        require(root.name == contract["task_id"], "单任务目录名不匹配")
    else:
        require(root.name in L46_STRATEGIES, "请提供课46根目录、策略目录或任务目录")
        groups[root.name] = (root, list(L46_TASK_IDS))
    report = {}
    for name, (base, ids) in groups.items():
        rows = [audit_lesson46_task(base / tid, tid) for tid in ids]
        successes = sum(r["record_supported_success"] for r in rows)
        proposals = sum(r["proposals"] or 0 for r in rows)
        report[name] = {"planned_tasks": len(ids), "record_supported_successes": successes,
                        "success_lower_bound": successes / len(ids), "rows": rows,
                        "observed_proposals": proposals,
                        "observed_proposals_per_success": proposals / successes if successes else None}
    return {"format": FORMAT, "mode": "lesson46_saved_evidence_audit", "groups": report,
            "candidate_execution_performed": False, "original_files_modified": False,
            "source_authenticity_proved": False, "sampling_metrics_available": False,
            "limits": ["只检查保存记录的一致性，不是独立重跑代码，也不能认证来源", "第46课多次自适应修复是一条trial，不按独立样本计算pass@k", "根目录分析按预先约定的三任务保留缺失分母"]}

In [24]:
# 填入你自己的第46课运行目录。None时明确跳过，不生成假的兼容成绩。
LESSON46_DIR = None
if LESSON46_DIR is None:
    print("未指定第46课结果；导入分析已跳过。")
else:
    imported = analyze_lesson46(LESSON46_DIR)
    print(json.dumps(imported, ensure_ascii=False, indent=2))

未指定第46课结果；导入分析已跳过。


## 9. 独立练习：先完成，再打开参考答案

本课检查独立于你的实现。下面三个练习函数都保留待完成内容，检查开关默认关闭。

**A：严格成功**。四个输入分别表示环境目标、最终回复、权限、记录有效；仅当全部为真正的 bool 且全部为 True 时成功。拒绝把数字 1 当作布尔。

**B：固定分母汇总**。输入计划 ID 和已返回的 `{"id":...,"success":...}`；缺失记为尚未验证成功，但保留分母；重复、未知 ID、非布尔成功标记必须拒绝。

**C：至少一次与全部成功**。计算有限样本的两个组合数比例，并检查 `n,c,k` 范围；返回与示例相同的五个字段。它不自动证明输入是独立同分布样本。

In [25]:
def check_exercise_a(fn: Callable) -> None:
    for a in (False, True):
        for b in (False, True):
            for c in (False, True):
                for d in (False, True):
                    require(fn(a, b, c, d) is (a and b and c and d), "A：成功条件不是全部满足")
    try: fn(1, True, True, True)
    except (ValueError, TypeError): pass
    else: raise ContractError("A：不要把整数1误当bool")


def check_exercise_b(fn: Callable) -> None:
    require(fn(["a", "b", "c"], [{"id": "a", "success": True}]) == 1 / 3, "B：分母应为计划数")
    require(fn(["a"], []) == 0, "B：全缺失仍保留分母")
    for planned, rows in [([], []), (["a", "a"], []), (["a"], [{"id": "b", "success": True}]),
                          (["a"], [{"id": "a", "success": 1}]),
                          (["a"], [{"id": "a", "success": True}] * 2)]:
        try: fn(planned, rows)
        except (ValueError, TypeError): pass
        else: raise ContractError("B：非法ID或记录没有被拒绝")


def check_exercise_c(fn: Callable) -> None:
    expected = sample_success_metrics(4, 2, 2)
    actual = fn(4, 2, 2)
    require(math.isclose(actual["at_least_one"], expected["at_least_one"]) and math.isclose(actual["all_k"], expected["all_k"]), "C：至少一次与全部成功混淆")
    for n, c, k in [(4, 0, 2), (4, 4, 2), (4, 2, 4), (1, 1, 1)]:
        require(fn(n, c, k) == sample_success_metrics(n, c, k), "C：边界错误")
    for args in [(2, 1, 3), (2, 3, 1), (0, 0, 1), (True, 1, 1), (3, 1, 0)]:
        try: fn(*args)
        except (ValueError, TypeError): pass
        else: raise ContractError("C：非法n/c/k没有拒绝")

In [26]:
def strict_success_ex(environment_goal, final_contract, policy_ok, valid_trace):
    # TODO: 校验bool类型；再取四个条件的交集。
    raise NotImplementedError("请完成练习A")

RUN_EXERCISE_A = False
if RUN_EXERCISE_A:
    check_exercise_a(strict_success_ex)
    print("练习A通过。")

In [27]:
def fixed_plan_rate_ex(planned, rows):
    # TODO: 检查计划与记录唯一；失败/缺失仍保留计划分母。
    raise NotImplementedError("请完成练习B")

RUN_EXERCISE_B = False
if RUN_EXERCISE_B:
    check_exercise_b(fixed_plan_rate_ex)
    print("练习B通过。")

In [28]:
def reliability_ex(n, c, k):
    # TODO: 返回 n,c,k,at_least_one,all_k；可使用math.comb。
    raise NotImplementedError("请完成练习C")

RUN_EXERCISE_C = False
if RUN_EXERCISE_C:
    check_exercise_c(reliability_ex)
    print("练习C通过。")

## 10. 参考答案与全套固定检查

参考答案使用独立函数名，不覆盖练习。下面的自动检查包括：伪造完成、幂等确认恢复、越权请求、坏 JSON、预算耗尽、轨迹篡改、缺失分母、指标公式、落盘重评分和第 46 课明细一致性。

**检查通过仅覆盖列出的情形，不证明任意系统安全或模型可靠。** 故障覆盖矩阵公开且有限，没有真实用户或模型采样，环境模型也很简单。

In [29]:
def strict_success_reference(environment_goal: bool, final_contract: bool, policy_ok: bool, valid_trace: bool) -> bool:
    require(all(type(x) is bool for x in (environment_goal, final_contract, policy_ok, valid_trace)), "需要布尔值")
    return environment_goal and final_contract and policy_ok and valid_trace


def fixed_plan_rate_reference(planned: list[str], rows: list[dict]) -> float:
    require(type(planned) is list and planned and all(type(x) is str for x in planned), "计划非空字符串列表")
    require(len(set(planned)) == len(planned), "重复计划ID")
    require(type(rows) is list and all(type(r) is dict and set(r) == {"id", "success"} for r in rows), "记录格式错误")
    require(all(type(r["id"]) is str and type(r["success"]) is bool for r in rows), "字段类型错误")
    require(len({r["id"] for r in rows}) == len(rows), "重复记录")
    require({r["id"] for r in rows} <= set(planned), "未知记录")
    return sum(r["success"] for r in rows) / len(planned)


def reliability_reference(n: int, c: int, k: int) -> dict:
    return sample_success_metrics(n, c, k)

In [30]:
def exercise_checks() -> dict:
    check_exercise_a(strict_success_reference)
    check_exercise_b(fixed_plan_rate_reference)
    check_exercise_c(reliability_reference)
    return {"A": True, "B": True, "C": True}

In [31]:
def run_checks() -> dict:
    names = []
    def check(name, condition):
        require(bool(condition), "检查失败：" + name)
        names.append(name)
    def raises(name, fn):
        try: fn()
        except (ContractError, ValueError, TypeError, KeyError): names.append(name)
        else: raise ContractError("未检出错误：" + name)
    tasks = make_tasks()
    plan = make_plan(tasks)
    check("八个固定任务", len(tasks) == 8)
    check("96个预先登记位置", len(plan["slots"]) == 96)
    check("每任务四个故障条件", len(PROFILES) == 4)
    raises("重复任务拒绝", lambda: make_plan(tasks + [tasks[0]]))
    bad = copy.deepcopy(tasks); bad[0]["goal"]["value"] += 1
    raises("任务目标一致性", lambda: make_plan(bad))
    raises("重复策略拒绝", lambda: make_plan(tasks, ["single_pass"] * 2))
    raises("负预算拒绝", lambda: make_plan(tasks, budget={**BUDGET, "max_turns": -1}))
    raises("bool预算拒绝", lambda: make_plan(tasks, budget={**BUDGET, "max_turns": True}))
    request = public_request(tasks[0], "trial")
    check("策略看不到oracle", "goal" not in request and "documents" not in request)
    check("策略看不到故障标签", "profile" not in request)
    altered = copy.deepcopy(tasks[0]); altered["goal"]["value"] = 999
    check("只改答案不改策略请求", public_request(altered, "trial") == request)
    raises("严格JSON拒绝重复键", lambda: strict_json('{"x":1,"x":2}'))
    raises("严格JSON拒绝NaN", lambda: strict_json('{"x":NaN}'))
    raises("最终数字不接受布尔", lambda: parse_action(canonical(final_action("completed", True, "T"))))
    raises("不能自己添加approved", lambda: parse_action(canonical({**final_action("failed"), "approved": True})))
    raises("未知工具拒绝", lambda: validate_tool("shell", {}))
    raises("计算器拒绝表达式", lambda: validate_tool("calculate", {"left": "1+2", "right": 1}))
    raises("计算器拒绝bool", lambda: validate_tool("calculate", {"left": True, "right": 1}))
    raises("工具额外字段拒绝", lambda: validate_tool("read_doc", {"doc_id": "left", "path": "/x"}))
    key = request["operation_key"]
    env = LocalEnvironment(tasks[0], "reply_lost", key)
    args = {"value": 16, "operation_key": key}
    first, second = env.invoke("create_ticket", args), env.invoke("create_ticket", args)
    check("确认丢失确实已提交", not first["ok"] and len(env.tickets) == 1)
    check("幂等重试返回原工单", second["ok"] and len(env.tickets) == 1)
    check("两次请求一次效果", env.requests == 2)
    check("同键不同参数拒绝", not env.invoke("create_ticket", {**args, "value": 17})["ok"])
    fresh = LocalEnvironment(tasks[0], "clean", key)
    check("每次试验新建服务", len(fresh.tickets) == 0)
    c = Controller(tasks[0], "clean", "test", {**BUDGET, "max_tool_attempts": 0})
    c.advance(canonical({"kind": "tool", "call_id": "x", "name": "read_doc", "arguments": {"doc_id": "left"}}))
    check("预算先检查后执行", c.stop == "tool_budget" and c.env.requests == 0)
    c = Controller(tasks[0], "clean", "test", {**BUDGET, "max_turns": 2})
    c.advance("bad"); c.advance("bad")
    check("坏JSON消耗决策轮次", c.stop == "turn_budget" and c.counts["decision_turns"] == 2)
    raises("终止后禁止追加", lambda: c.advance("bad"))
    c = Controller(tasks[0], "clean", "test", BUDGET)
    act = canonical({"kind": "tool", "call_id": "x", "name": "read_doc", "arguments": {"doc_id": "left"}})
    c.advance(act); c.advance(act)
    check("重复调用ID不执行第二遍", c.env.requests == 1 and c.events[1]["observation"]["error"]["code"] == "duplicate_call_id")
    all_records = [run_trial(next(t for t in tasks if t["task_id"] == s["task_id"]), s, plan) for s in plan["slots"]]
    report = score_records(tasks, plan, all_records)
    summaries = report["strategies"]
    check("宣布完成策略无真实成功", summaries["announce_only"]["strict_successes"] == 0)
    check("有界重试覆盖全部条件", summaries["bounded_retry"]["strict_successes"] == 32)
    check("单次策略预期13次成功", summaries["single_pass"]["strict_successes"] == 13)
    check("成本未测不能用零", summaries["bounded_retry"]["tokens"] is None)
    check("零成功成本比无定义", summaries["announce_only"]["observed_tool_attempts_per_success"] is None)
    check("固定条件不报告pass_at_k", summaries["single_pass"]["not_pass_at_k"])
    by_slot = {s["trial_id"]: s for s in plan["slots"]}
    by_record = {r["trial_id"]: r for r in all_records}
    tid = "single_pass::ticket_01::reply_lost"
    g = grade_trial(tasks[0], by_slot[tid], plan, by_record[tid])
    check("服务完成与用户确认分开", g["environment_goal"] and not g["strict_success"])
    check("确认丢失诊断", g["diagnosis"] == "outcome_present_but_completion_not_confirmed")
    tid2 = "bounded_retry::ticket_01::reply_lost"
    r = by_record[tid2]
    check("恢复有额外分派成本", r["counts"]["tool_attempts"] == 5 and len(r["outcome"]["tickets"]) == 1)
    g2 = grade_trial(tasks[0], by_slot[tid2], plan, r)
    check("成功轨迹也保留首个错误", g2["strict_success"] and g2["first_observed_error"] is not None)
    check("相同输入确定性重演", replay_and_verify(tasks[0], by_slot[tid2], plan, r).record(by_slot[tid2], plan["plan_sha256"]) == r)
    for label, mutate in [
        ("虚构环境结果", lambda x: x["outcome"]["tickets"][0].update(value=999)),
        ("虚报调用计数", lambda x: x["counts"].update(tool_attempts=0)),
        ("假成功结果", lambda x: x["final"].update(value=999)),
        ("工具观察被改", lambda x: x["events"][0]["observation"]["data"].update(days=999)),
        ("事件顺序被改", lambda x: x["events"].reverse()),
        ("模板或协议改动", lambda x: x.update(plan_sha256="other")),
    ]:
        bad_record = copy.deepcopy(r); mutate(bad_record)
        check(label, grade_trial(tasks[0], by_slot[tid2], plan, bad_record)["diagnosis"] == "invalid_trace")
    missing = score_records(tasks, plan, [r for r in all_records if r["trial_id"] != tid2])
    check("缺失记录仍是32分母", missing["strategies"]["bounded_retry"]["strict_success_rate_lower_bound"] == 31 / 32)
    check("缺失状态单独报告", missing["strategies"]["bounded_retry"]["missing_records"] == 1)
    raises("重复结果不能增加成功", lambda: score_records(tasks, plan, all_records + [r]))
    unknown = copy.deepcopy(r); unknown["trial_id"] = "unknown"
    raises("额外题目不能刷分", lambda: score_records(tasks, plan, [unknown]))
    comparison = paired_comparison(report)
    check("配对改善19个条件", len(comparison["fixed"]) == 19)
    check("配对退步为空", comparison["regressed"] == [])
    unsafe_slot = {"trial_id": "bounded_retry::refuse_permission::clean", "strategy": "bounded_retry", "task_id": "refuse_permission", "profile": "clean"}
    denied = tasks[-2]
    calls = iter([canonical({"kind": "tool", "call_id": "bad", "name": "create_ticket", "arguments": {"value": 16, "operation_key": public_request(denied, unsafe_slot["trial_id"])["operation_key"]}}), canonical(final_action("abstain", reason="not_authorized"))])
    rr = run_trial(denied, unsafe_slot, plan, lambda view: next(calls))
    gg = grade_trial(denied, unsafe_slot, plan, rr)
    check("拒绝工具仍有违规尝试", gg["environment_goal"] and gg["final_contract"] and not gg["strict_success"])
    check("不得把拒绝当工具副作用", rr["outcome"]["tickets"] == [])
    for n, c0, k in [(4, 2, 2), (4, 0, 4), (4, 4, 4), (4, 2, 1), (4, 2, 4)]:
        z = sample_success_metrics(n, c0, k)
        check(f"样本指标边界{n}_{c0}_{k}", 0 <= z["all_k"] <= z["at_least_one"] <= 1)
    z = sample_success_metrics(4, 2, 2)
    check("至少一次5/6", math.isclose(z["at_least_one"], 5 / 6))
    check("全部成功1/6", math.isclose(z["all_k"], 1 / 6))
    raises("不能k大于n", lambda: sample_success_metrics(2, 1, 3))
    raises("不能布尔样本数", lambda: sample_success_metrics(True, 1, 1))
    check("微宏平均有区别", mechanisms()["micro_macro"]["micro"] != mechanisms()["micro_macro"]["macro"])
    check("成功条件全部满足", strict_success_reference(True, True, True, True))
    check("成功不取决于工具全成功", g2["strict_success"] and r["counts"]["tool_successes"] < r["counts"]["tool_executions"])
    block = {"total": 1, "passed": 1, "all_pass": True, "records": [{"id": "x", "numbers": [], "expected": 0, "actual": 0, "passed": True, "error": None}]}
    check("课46明细检查", audit_case_report(block)[0])
    bad_block = copy.deepcopy(block); bad_block["records"][0]["actual"] = 1
    raises("不能相信原始passed", lambda: audit_case_report(bad_block))
    bad_block = copy.deepcopy(block); bad_block["records"][0]["actual"] = False
    raises("布尔不能当整数结果", lambda: audit_case_report(bad_block))
    check("文件缺失不是成功", not audit_lesson46_task(Path("does_not_exist_lesson47"), "sum_nonnegative")["record_supported_success"])
    with tempfile.TemporaryDirectory(prefix="lesson47_checks_") as tmp:
        run = run_experiment(tmp)
        before = {str(p.relative_to(run)): p.read_bytes() for p in run.rglob("*") if p.is_file()}
        rescored = rescore(run)
        check("磁盘重新评分一致", rescored == read_json(run / "report.json"))
        check("重新评分未写原文件", before == {str(p.relative_to(run)): p.read_bytes() for p in run.rglob("*") if p.is_file()})
        lines = (run / "trials.jsonl").read_text(encoding="utf-8").splitlines()
        (run / "trials.jsonl").write_text("\n".join(lines[:-1]) + "\n", encoding="utf-8")
        check("删除整行保持计划分母", rescore(run)["strategies"]["bounded_retry"]["missing_records"] == 1)
        (run / "trials.jsonl").write_text("broken\n", encoding="utf-8")
        raises("损坏JSONL不静默跳过", lambda: rescore(run))
    exercise_checks()
    check("三组练习参考通过", True)
    return {"passed": len(names), "checks": names, "scope": "标准库模拟与评测协议；不是LLM、GPU或真实业务服务测试"}

In [32]:
checks = run_checks()
print("固定检查通过：", checks["passed"], "项")
print("参考答案检查：", exercise_checks())
if run_dir is not None:
    write_json(run_dir / "checks.json", checks)
    write_json(run_dir / "exercise_checks.json", exercise_checks())
print("范围：标准库模拟；没有模型、GPU或外部服务实测。")

固定检查通过： 74 项
参考答案检查： {'A': True, 'B': True, 'C': True}
范围：标准库模拟；没有模型、GPU或外部服务实测。


## 11. 命令行与验收

```bash
python lesson_47_agent_evaluation_traces_and_budgets.py --checks-only
python lesson_47_agent_evaluation_traces_and_budgets.py
python lesson_47_agent_evaluation_traces_and_budgets.py --rescore 你的本课运行目录
python lesson_47_agent_evaluation_traces_and_budgets.py --analyze-lesson46 你的第46课运行目录
```

主输出包括 `plan.json`、`tasks.json`、`trials.jsonl`、`report.json`、`comparison.json`、`mechanisms.json`、`manifest.json`。明文记录中可能包含任务数据，分享前检查隐私。

**本课验收：** 能解释为何“工单已存在”仍可能没有完成交付；能解释缺失记录为何仍留在计划分母；能从一条轨迹重新算出各类调用次数；能区分一次 trial 内重试与多次独立 trial；能指出第 46 课导入器没有重新执行代码。

### 下一课

**第 48 课：推理服务接口、请求生命周期与端到端资源预算。** 把现有模型／Agent 包装成可调用服务，继续处理输入约束、请求 ID、超时与取消、结构化日志，以及怎样把本课回归集用于接口变更验证。

## 主要资料（原始论文与官方文档）

1. [Anthropic：Demystifying evals for AI agents](https://www.anthropic.com/engineering/demystifying-evals-for-ai-agents)——task、trial、trace、outcome及评分分层。
2. [τ-bench 原论文](https://arxiv.org/abs/2406.12045)——工具交互终态与重复可靠性指标。
3. [HumanEval 官方评分实现](https://github.com/openai/human-eval/blob/master/human_eval/evaluation.py)——pass@k组合数估计。
4. [SWE-bench Evaluation Guide](https://www.swebench.com/SWE-bench/guides/evaluation/)——完成、解决、缺失与环境错误的不同口径。
5. [Anthropic：A statistical approach to model evaluations](https://www.anthropic.com/research/statistical-approach-to-model-evals)——配对、相关性与不确定性。
6. [Python JSON](https://docs.python.org/3/library/json.html) 与 [statistics](https://docs.python.org/3/library/statistics.html)——输入解析和标准库统计工具。

程序中的任务、故障条件和数值结果来自本课执行，不来自这些资料的模型榜单。